# Aggregated CNN–LSTM–Attention–Residual Model

Full pipeline: Data loading → Encoding → k-fold training (dual hybrid loss) → Dual-level evaluation → Model saving.

**Repository:** https://github.com/parkingvarsson/Aggregated-DL

## 1. Dependencies and Hyperparameters

In [ ]:
import os, math, pickle
from datetime import datetime
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW, lr_scheduler
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEQ_LENGTH          = 40
ORIGINAL_SEQ_LENGTH = 280
BATCH_SIZE = 256
EPOCHS     = 100
PATIENCE   = 10
K_FOLDS    = 3
ALPHA      = 0.7

DATA_FOLDER_40nt  = '/content/drive/MyDrive/.../40nt'
DATA_FOLDER_200nt = '/content/drive/MyDrive/.../200nt'
SAVE_DIR          = '/content/drive/MyDrive/model_output'


## 2. Data Loading and Encoding

### 2.1 Sequence-Information Container

In [ ]:
class OriginalSequenceInfo:
    # Bidirectional index: original genes <-> augmented subsequences
    def __init__(self):
        self.original_to_augmented = defaultdict(list)
        self.augmented_to_original = {}
        self.original_sequences    = []
        self.original_labels       = []
        self.original_gene_ids     = []
        self.augmented_sequences   = []
        self.augmented_positions   = []

    def add_original_sequence(self, sequence, label, gene_id):
        self.original_sequences.append(sequence)
        self.original_labels.append(label)
        self.original_gene_ids.append(gene_id)

    def add_augmented_sequence(self, sequence, start_pos=None, end_pos=None):
        self.augmented_sequences.append(sequence)
        self.augmented_positions.append((start_pos, end_pos))

    def add_mapping(self, original_idx, augmented_indices, positions=None):
        self.original_to_augmented[original_idx].extend(augmented_indices)
        for i, aug_idx in enumerate(augmented_indices):
            self.augmented_to_original[aug_idx] = original_idx
            if positions and i < len(positions):
                self.augmented_positions[aug_idx] = positions[i]

    def get_augmented_for_original(self, original_idx):
        return self.original_to_augmented.get(original_idx, [])


### 2.2 Augmentation-Mapping Validation

Prints a concise summary only; verbose per-subsequence debug output removed.

In [ ]:
def validate_augmentation_mapping(original_info):
    mismatch_count = 0
    position_errors = 0
    for orig_idx in range(len(original_info.original_sequences)):
        original_seq = original_info.original_sequences[orig_idx]
        for aug_idx in original_info.get_augmented_for_original(orig_idx):
            aug_seq            = original_info.augmented_sequences[aug_idx]
            start_pos, end_pos = original_info.augmented_positions[aug_idx]
            if start_pos is None or end_pos is None:
                mismatch_count += 1; continue
            if end_pos > len(original_seq) or start_pos < 0:
                position_errors += 1; continue
            if original_seq[start_pos:end_pos] != aug_seq:
                mismatch_count += 1
    print('\nValidation Summary:')
    print(f'  Total original sequences : {len(original_info.original_sequences)}')
    print(f'  Total augmented sequences: {len(original_info.augmented_sequences)}')
    print(f'  Position errors          : {position_errors}')
    print(f'  Sequence mismatches      : {mismatch_count}')
    if position_errors == 0 and mismatch_count == 0:
        print('  All augmented sequences correctly map to their parent sequences.')
        return True
    print('  WARNING: Mapping errors detected.')
    return False


### 2.3 One-Hot Encoding

Five channels: A=0, T=1, C=2, G=3, N=4. Binary mask marks valid (non-N) positions.

In [ ]:
def one_hot_encode(sequence, seq_length=SEQ_LENGTH):
    if not isinstance(sequence, str):
        sequence = str(sequence)
    nucleotide_map = {
        'A':[1,0,0,0,0],'T':[0,1,0,0,0],
        'C':[0,0,1,0,0],'G':[0,0,0,1,0],'N':[0,0,0,0,1]}
    sequence   = sequence.upper()
    valid_mask = np.ones(len(sequence), dtype=np.float32)
    if len(sequence) < seq_length:
        sequence   = sequence.ljust(seq_length, 'N')
        valid_mask = np.pad(valid_mask,
                            (0, seq_length - len(sequence)), constant_values=0)
    else:
        sequence   = sequence[:seq_length]
        valid_mask = valid_mask[:seq_length]
    encoded = np.array([nucleotide_map.get(c,[0,0,0,0,1]) for c in sequence])
    return encoded, valid_mask


### 2.4 Data Loading

In [ ]:
def load_data(data_folder_40nt, data_folder_200nt):
    class_names   = sorted(f[:-4] for f in os.listdir(data_folder_40nt)
                           if f.endswith('.csv'))
    original_info = OriginalSequenceInfo()
    raw_sequences, labels = [], []

    for class_idx, class_name in enumerate(class_names):
        path = os.path.join(data_folder_200nt, f'{class_name}.csv')
        orig_data = pd.read_csv(path, header=None)
        for idx, row in orig_data.iterrows():
            original_info.add_original_sequence(
                row[0], class_idx, f'{class_name}_{idx}')

    current_aug_idx  = 0
    n_orig_per_class = len(original_info.original_sequences) // len(class_names)

    for class_idx, class_name in enumerate(class_names):
        path     = os.path.join(data_folder_40nt, f'{class_name}.csv')
        aug_data = pd.read_csv(path, header=None)
        for orig_idx in range(n_orig_per_class):
            subsequences    = aug_data.iloc[orig_idx*240:(orig_idx+1)*240, 0].tolist()
            global_orig_idx = class_idx * n_orig_per_class + orig_idx
            original_seq    = original_info.original_sequences[global_orig_idx]
            positions = []
            for seq in subsequences:
                pos = original_seq.find(seq)
                positions.append((pos, pos+len(seq)) if pos != -1 else (None, None))
                original_info.add_augmented_sequence(seq, *positions[-1])
            raw_sequences.extend(subsequences)
            labels.extend([class_idx] * 240)
            original_info.add_mapping(
                global_orig_idx,
                range(current_aug_idx, current_aug_idx + 240), positions)
            current_aug_idx += 240

    if not validate_augmentation_mapping(original_info):
        raise ValueError('Augmentation mapping validation failed.')

    one_hot_seqs, valid_masks = [], []
    for seq in raw_sequences:
        enc, msk = one_hot_encode(seq)
        one_hot_seqs.append(enc); valid_masks.append(msk)

    return (np.array(one_hot_seqs), np.array(valid_masks),
            np.array(labels), class_names, original_info)


### 2.5 Gene Mapping and Coverage Analysis

In [ ]:
def create_gene_mapping(data_folder_40nt, class_names):
    aug_to_gene, gene_to_aug = {}, defaultdict(list)
    current_idx = 0
    for class_idx, class_name in enumerate(class_names):
        path = os.path.join(data_folder_40nt, f'{class_name}.csv')
        with open(path) as f:
            for line in f:
                parts = line.strip().rsplit(',', 1)
                if len(parts) == 2:
                    gene_id = parts[1].strip()
                    aug_to_gene[current_idx] = (gene_id, class_idx)
                    gene_to_aug[(gene_id, class_idx)].append(current_idx)
                    current_idx += 1
    return {'aug_to_gene': aug_to_gene, 'gene_to_aug': gene_to_aug}


def compute_coverage(original_info, original_seq_length=200):
    total_orig   = len(original_info.original_sequences)
    pos_coverage = np.zeros(original_seq_length)
    for orig_idx in range(total_orig):
        for aug_idx in original_info.get_augmented_for_original(orig_idx):
            start_pos, end_pos = original_info.augmented_positions[aug_idx]
            if start_pos is None:
                continue
            s = max(0, start_pos - 40)
            e = min(original_seq_length - 1, end_pos - 40)
            for p in range(s, e + 1):
                if 0 <= p < original_seq_length:
                    pos_coverage[p] += 1
    sufficient = bool(np.min(pos_coverage) > 0)
    print(f'  Average coverage per position : {np.mean(pos_coverage):.2f}')
    print(f'  Minimum coverage              : {np.min(pos_coverage):.0f}')
    print(f'  Coverage sufficient           : {sufficient}')
    return {'position_coverage': pos_coverage,
            'avg_coverage': float(np.mean(pos_coverage)),
            'sufficient_coverage': sufficient}


## 3. PyTorch Dataset

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, sequences, masks, labels):
        self.sequences = sequences
        self.masks     = masks
        self.labels    = labels
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return (torch.as_tensor(self.sequences[idx], dtype=torch.float32),
                torch.as_tensor(self.masks[idx],     dtype=torch.float32),
                torch.tensor(self.labels[idx],        dtype=torch.long))


## 4. Model Architecture

| Component | Details |
|---|---|
| CNN Block 1 | Conv1D (128 filters, k=3) + BN + MaxPool + Dropout(0.3) + Residual + CNN-Attention |
| CNN Block 2 | Multi-kernel Conv1D (k=3,5,7; 256 each → 768 ch) + BN + MaxPool + Dropout(0.5) + Residual + CNN-Attention |
| CNN Block 3 | Multi-kernel Conv1D (k=3,5,7; 512 each → 1536 ch) + BN + MaxPool + Dropout(0.3) + Residual + CNN-Attention |
| Bi-LSTM | 2 layers, hidden=256/direction (total 512) + Dropout(0.3) + LSTM-Attention |
| FC layers | FC1(256) → FC2(512) → FC3(512) → FC4(num_classes) |

### 4.1 Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    def forward(self, x):
        return x + self.pe[:x.size(1)]


### 4.2 CNN Block with Residual Connections

In [ ]:
class MultiKernelCNN(nn.Module):
    def __init__(self, input_channels, output_channels,
                 use_multi_kernel=True, dropout_rate=0.3):
        super().__init__()
        self.use_multi_kernel = use_multi_kernel
        factor = 3 if use_multi_kernel else 1
        if use_multi_kernel:
            self.conv3 = nn.Conv1d(input_channels, output_channels, 3, padding=1)
            self.conv5 = nn.Conv1d(input_channels, output_channels, 5, padding=2)
            self.conv7 = nn.Conv1d(input_channels, output_channels, 7, padding=3)
        else:
            self.conv3 = nn.Conv1d(input_channels, output_channels, 3, padding=1)
        self.relu    = nn.ReLU()
        self.pool    = nn.MaxPool1d(kernel_size=2, stride=2)
        self.bn      = nn.BatchNorm1d(output_channels * factor)
        self.dropout = nn.Dropout(dropout_rate)
        if input_channels != output_channels * factor:
            self.residual = nn.Sequential(
                nn.Conv1d(input_channels, output_channels * factor, 1),
                nn.BatchNorm1d(output_channels * factor))
        else:
            self.residual = nn.Sequential()
    def forward(self, x):
        identity = self.residual(x)
        if self.use_multi_kernel:
            x = torch.cat([self.relu(self.conv3(x)),
                           self.relu(self.conv5(x)),
                           self.relu(self.conv7(x))], dim=1)
        else:
            x = self.relu(self.conv3(x))
        x = self.bn(x); x = self.pool(x); x = self.dropout(x)
        if identity.size(-1) > x.size(-1):
            identity = identity[..., :x.size(-1)]
        elif identity.size(-1) < x.size(-1):
            identity = F.pad(identity, (0, x.size(-1) - identity.size(-1)))
        return x + identity


### 4.3 Attention Modules

In [ ]:
class CNN_Attention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction_ratio, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction_ratio, in_channels, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        b, c, _ = x.size()
        ca  = self.fc(self.avg_pool(x).view(b,c)) + self.fc(self.max_pool(x).view(b,c))
        spa = torch.mean(x, dim=1, keepdim=True)
        return x * ca.view(b,c,1) * spa, ca, spa


class LSTM_Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1))
    def forward(self, lstm_output):
        weights = F.softmax(
            self.attention(lstm_output).squeeze(-1), dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights


### 4.4 Full Model

In [ ]:
class Optimized_CNN_LSTM_Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn1           = MultiKernelCNN(5,     128, use_multi_kernel=False, dropout_rate=0.3)
        self.cnn1_attention  = CNN_Attention(128)
        self.cnn2           = MultiKernelCNN(128,   256, use_multi_kernel=True,  dropout_rate=0.5)
        self.cnn2_attention  = CNN_Attention(256*3)
        self.cnn3           = MultiKernelCNN(256*3, 512, use_multi_kernel=True,  dropout_rate=0.3)
        self.cnn3_attention  = CNN_Attention(512*3)
        self.pos_encoder    = PositionalEncoding(d_model=512*3)
        self.lstm           = nn.LSTM(
            input_size=512*3, hidden_size=256, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3)
        self.lstm_attention  = LSTM_Attention(hidden_size=512)
        self.attention_weights = {
            'cnn1':None,'cnn2':None,'cnn3':None,'spatial':None,'lstm':None}
        self.fc1 = nn.Linear(512, 256); self.bn_fc1 = nn.BatchNorm1d(256); self.dropout_fc1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 512); self.bn_fc2 = nn.BatchNorm1d(512); self.dropout_fc2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(512, 512); self.bn_fc3 = nn.BatchNorm1d(512); self.dropout_fc3 = nn.Dropout(0.3)
        self.fc4  = nn.Linear(512, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x, mask=None):
        if mask is not None:
            x = x * mask.unsqueeze(-1)
        x = x.permute(0, 2, 1)
        x = self.cnn1(x);  x, self.attention_weights['cnn1'], _                           = self.cnn1_attention(x)
        x = self.cnn2(x);  x, self.attention_weights['cnn2'], _                           = self.cnn2_attention(x)
        x = self.cnn3(x);  x, self.attention_weights['cnn3'], self.attention_weights['spatial'] = self.cnn3_attention(x)
        x = x.permute(0, 2, 1)
        x = self.pos_encoder(x)
        lstm_out, _        = self.lstm(x)
        context, lstm_attn = self.lstm_attention(lstm_out)
        self.attention_weights['lstm'] = lstm_attn
        x = self.relu(self.dropout_fc1(self.bn_fc1(self.fc1(context))))
        x = self.relu(self.dropout_fc2(self.bn_fc2(self.fc2(x))))
        x = self.relu(self.dropout_fc3(self.bn_fc3(self.fc3(x))))
        return self.fc4(x)

    def get_attention_weights(self):
        return self.attention_weights


## 5. Training

### 5.1 Dual Hybrid Loss Training Step

Total loss = α · L_subseq + (1−α) · L_gene  (α = 0.7)

In [ ]:
def train_with_gene_loss(model, train_loader, optimizer, criterion,
                         gene_mapping, device, scaler, scheduler, alpha=ALPHA):
    model.train()
    train_loss = gene_loss_total = 0.0
    correct = total = 0
    unique_genes = sorted({g for (g, _) in gene_mapping['gene_to_aug']})
    gene_to_idx  = {g: i for i, g in enumerate(unique_genes)}

    for batch_idx, (data, mask, target) in enumerate(train_loader):
        data, mask, target = data.to(device), mask.to(device), target.to(device)
        batch_start  = batch_idx * BATCH_SIZE
        gene_indices = []
        for idx in range(batch_start, batch_start + len(data)):
            gene_id, _ = gene_mapping['aug_to_gene'].get(idx, (f'dummy_{idx}', 0))
            gene_indices.append(gene_to_idx.get(gene_id, len(unique_genes)))
        gene_indices = torch.tensor(gene_indices, device=device)

        optimizer.zero_grad()
        outputs     = model(data, mask)
        subseq_loss = criterion(outputs, target)

        unique_g, _ = torch.unique(gene_indices, return_inverse=True)
        gene_loss   = torch.tensor(0.0, device=device)
        valid_genes = 0
        for gene in unique_g:
            g_mask    = (gene_indices == gene)
            g_outputs = outputs[g_mask]
            g_targets = target[g_mask]
            if len(g_outputs) < 2:
                continue
            avg_pred  = g_outputs.mean(dim=0, keepdim=True)
            gene_loss = gene_loss + criterion(avg_pred, g_targets[:1])
            valid_genes += 1

        if valid_genes > 0:
            gene_loss       = gene_loss / valid_genes
            gene_loss_total += gene_loss.item()
            loss = alpha * subseq_loss + (1 - alpha) * gene_loss
        else:
            loss = subseq_loss

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total   += target.size(0)
        correct += predicted.eq(target).sum().item()

    return (train_loss / len(train_loader),
            100 * correct / total,
            gene_loss_total / len(train_loader) if gene_loss_total > 0 else 0.0)


### 5.2 Validation Step

In [ ]:
def validate(model, val_loader, criterion):
    model.eval()
    val_loss = correct = total = 0
    with torch.no_grad():
        for data, mask, target in val_loader:
            data, mask, target = data.to(device), mask.to(device), target.to(device)
            outputs   = model(data, mask)
            val_loss += criterion(outputs, target).item()
            _, predicted = outputs.max(1)
            total   += target.size(0)
            correct += predicted.eq(target).sum().item()
    return val_loss / len(val_loader), 100 * correct / total


## 6. Evaluation

### 6.1 Subsequence-Level Evaluation

In [ ]:
def evaluate_model(model, data_loader, device, class_names):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for data, mask, target in data_loader:
            data, mask, target = data.to(device), mask.to(device), target.to(device)
            outputs = model(data, mask)
            all_probs.append(F.softmax(outputs, dim=1).cpu().numpy())
            all_labels.append(target.cpu().numpy())
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    aug_preds  = np.argmax(all_probs, axis=1)
    print('\nAugmented Subsequence-Level Evaluation:')
    print(classification_report(all_labels, aug_preds,
                                 target_names=class_names, digits=4))
    return {'probs': all_probs, 'labels': all_labels, 'preds': aug_preds}


### 6.2 Original Sequence-Level Evaluation

Attention-weighted aggregation of FC3 features across all 240 subsequences per gene (Equation 4, manuscript).

In [ ]:
def evaluate_gene_level(model, sequences, masks, original_info, class_names, device):
    model.eval()
    all_gene_probs, all_gene_labels = [], []

    for orig_idx in range(len(original_info.original_sequences)):
        aug_indices = original_info.get_augmented_for_original(orig_idx)
        if not aug_indices:
            continue
        gene_features, gene_attn_weights = [], []

        for i in range(0, len(aug_indices), BATCH_SIZE):
            batch_idx   = aug_indices[i:i + BATCH_SIZE]
            batch_data  = torch.stack([
                torch.tensor(sequences[j], dtype=torch.float32)
                for j in batch_idx]).to(device)
            batch_masks = torch.stack([
                torch.tensor(masks[j], dtype=torch.float32)
                for j in batch_idx]).to(device)
            with torch.no_grad():
                x = batch_data * batch_masks.unsqueeze(-1)
                x = x.permute(0, 2, 1)
                x = model.cnn1(x);  x, _, _ = model.cnn1_attention(x)
                x = model.cnn2(x);  x, _, _ = model.cnn2_attention(x)
                x = model.cnn3(x);  x, _, _ = model.cnn3_attention(x)
                x = x.permute(0, 2, 1)
                x = model.pos_encoder(x)
                lstm_out, _       = model.lstm(x)
                context, attn_w   = model.lstm_attention(lstm_out)
                feat = model.relu(model.dropout_fc1(model.bn_fc1(model.fc1(context))))
                feat = model.relu(model.dropout_fc2(model.bn_fc2(model.fc2(feat))))
                feat = model.relu(model.dropout_fc3(model.bn_fc3(model.fc3(feat))))
                gene_features.append(feat.cpu().numpy())
                gene_attn_weights.append(attn_w.cpu().numpy())

        if gene_features:
            all_feat = np.concatenate(gene_features)
            all_attn = np.concatenate(gene_attn_weights)
            scalar_w = np.mean(all_attn, axis=1)
            if scalar_w.sum() > 0:
                scalar_w = scalar_w / scalar_w.sum()
                agg_feat = np.average(all_feat, axis=0, weights=scalar_w)
            else:
                agg_feat = np.mean(all_feat, axis=0)
            agg_tensor = torch.tensor(
                agg_feat, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model.fc4(agg_tensor)
                probs  = F.softmax(logits, dim=1).cpu().numpy()[0]
            all_gene_probs.append(probs)
            all_gene_labels.append(original_info.original_labels[orig_idx])

    gene_preds = np.argmax(all_gene_probs, axis=1)
    print('\nOriginal Full-Length Sequence-Level Evaluation:')
    print(classification_report(all_gene_labels, gene_preds,
                                 target_names=class_names, digits=4))
    return {'probs': np.array(all_gene_probs),
            'labels': np.array(all_gene_labels),
            'preds': gene_preds}


## 7. Model Serialisation

In [ ]:
def save_model(model, class_names, original_info, coverage_results,
               sequences, masks, hyperparams=None, save_dir=SAVE_DIR):
    os.makedirs(save_dir, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    def pkl(obj, name):
        path = os.path.join(save_dir, f'{name}_{ts}.pkl')
        with open(path, 'wb') as f: pickle.dump(obj, f)
        return path
    model_path = os.path.join(save_dir, f'model_{ts}.pt')
    torch.save(model.state_dict(), model_path)
    cn_path = os.path.join(save_dir, f'class_names_{ts}.txt')
    with open(cn_path, 'w') as f: f.write('\n'.join(class_names))
    seq_path  = os.path.join(save_dir, f'sequences_{ts}.npz')
    mask_path = os.path.join(save_dir, f'masks_{ts}.npz')
    np.savez_compressed(seq_path,  sequences=sequences)
    np.savez_compressed(mask_path, masks=masks)
    metadata = {
        'timestamp'       : ts,
        'model_path'      : model_path,
        'class_names_path': cn_path,
        'original_info'   : pkl(original_info,    'original_info'),
        'coverage_results': pkl(coverage_results, 'coverage_results'),
        'sequences_path'  : seq_path,
        'masks_path'      : mask_path,
        'hyperparams'     : pkl(hyperparams, 'hyperparams') if hyperparams else None,
    }
    meta_path = pkl(metadata, 'metadata')
    print(f'\nModel package saved (timestamp: {ts})')
    print(f'Metadata index: {meta_path}')
    return metadata


## 8. Main Training and Evaluation Pipeline

Data loading → coverage check → stratified k-fold CV → dual-level evaluation → model saving.

In [ ]:
def main():
    print('Loading data...')
    all_sequences, all_masks, labels, class_names, original_info = load_data(
        DATA_FOLDER_40nt, DATA_FOLDER_200nt)
    print(f'Classes        : {class_names}')
    print(f'Total subseqs  : {len(all_sequences)}')
    print(f'Total genes    : {len(original_info.original_sequences)}')

    print('\nComputing positional coverage...')
    coverage_results = compute_coverage(original_info)
    if not coverage_results['sufficient_coverage']:
        raise RuntimeError('Insufficient coverage.')

    gene_mapping = create_gene_mapping(DATA_FOLDER_40nt, class_names)
    skf          = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    fold_states  = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(all_sequences, labels)):
        print(f"\n{'='*60}\n  Fold {fold+1} / {K_FOLDS}\n{'='*60}")
        X_tr, X_te = all_sequences[train_idx], all_sequences[test_idx]
        M_tr, M_te = all_masks[train_idx],     all_masks[test_idx]
        y_tr, y_te = labels[train_idx],         labels[test_idx]
        X_tr, X_val, M_tr, M_val, y_tr, y_val = train_test_split(
            X_tr, M_tr, y_tr, test_size=0.2, random_state=42, stratify=y_tr)

        pin = (device.type == 'cuda')
        train_loader = DataLoader(SequenceDataset(X_tr,  M_tr,  y_tr),
                                   batch_size=BATCH_SIZE, shuffle=True,
                                   num_workers=2, pin_memory=pin)
        val_loader   = DataLoader(SequenceDataset(X_val, M_val, y_val),
                                   batch_size=BATCH_SIZE, shuffle=False,
                                   num_workers=2, pin_memory=pin)
        test_loader  = DataLoader(SequenceDataset(X_te,  M_te,  y_te),
                                   batch_size=BATCH_SIZE, shuffle=False)

        model     = Optimized_CNN_LSTM_Model(num_classes=len(class_names)).to(device)
        optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        scheduler = lr_scheduler.OneCycleLR(
            optimizer, max_lr=1e-3, epochs=EPOCHS,
            steps_per_epoch=len(train_loader), pct_start=0.3)
        scaler = (torch.amp.GradScaler('cuda')
                  if device.type == 'cuda' else None)

        best_val_loss  = float('inf')
        patience_count = 0

        for epoch in range(EPOCHS):
            tr_loss, tr_acc, g_loss = train_with_gene_loss(
                model, train_loader, optimizer, criterion,
                gene_mapping, device, scaler, scheduler)
            val_loss, val_acc = validate(model, val_loader, criterion)
            print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
                  f'Train {tr_loss:.4f} (gene {g_loss:.4f}) {tr_acc:.1f}% | '
                  f'Val {val_loss:.4f} {val_acc:.1f}%')
            if val_loss < best_val_loss:
                best_val_loss  = val_loss
                patience_count = 0
            else:
                patience_count += 1
                if patience_count >= PATIENCE:
                    print(f'  Early stopping at epoch {epoch+1}.')
                    break

        evaluate_model(model, test_loader, device, class_names)
        fold_states.append({k: v.clone() for k, v in model.state_dict().items()})

    print(f"\n{'='*60}\n  FINAL EVALUATION (last-fold model)\n{'='*60}")
    final_model = Optimized_CNN_LSTM_Model(num_classes=len(class_names)).to(device)
    final_model.load_state_dict(fold_states[-1])
    full_loader = DataLoader(
        SequenceDataset(all_sequences, all_masks, labels),
        batch_size=BATCH_SIZE, shuffle=False)
    evaluate_model(final_model, full_loader, device, class_names)
    evaluate_gene_level(final_model, all_sequences, all_masks,
                        original_info, class_names, device)

    hyperparams = {'SEQ_LENGTH': SEQ_LENGTH, 'BATCH_SIZE': BATCH_SIZE,
                   'EPOCHS': EPOCHS, 'K_FOLDS': K_FOLDS, 'ALPHA': ALPHA}
    save_model(final_model, class_names, original_info, coverage_results,
               all_sequences, all_masks, hyperparams=hyperparams, save_dir=SAVE_DIR)


main()
